# H-GNN-RNN-DTI - Objective 1 Fusion Ablation (GPU)

Runs the core RQ1 experiment: **GCA vs concat vs flat_gate vs ungated_xattn** on DAVIS (S1 warm-start), 3 seeds each, reporting AUROC/AUPR mean±std + a paired significance test.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator = GPU (T4)`, then `Runtime -> Run all`.

If the `git clone` cell fails, the code hasn't been pushed to the branch yet - push `objective1-impl` to `https://github.com/jamcy98/Modeling-DTI.git`, or upload a zip of the `H-GNN-RNN-DTI/` folder and `%cd` into it.


## 1. Check the GPU


In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Get the code


In [ ]:
REPO_URL = 'https://github.com/jamcy98/Modeling-DTI.git'
BRANCH = 'objective1-impl'
!git clone --depth 1 --branch $BRANCH $REPO_URL
%cd Modeling-DTI/H-GNN-RNN-DTI


## 3. Install dependencies
(Colab already ships a CUDA build of PyTorch; we add the rest. `PyTDC==0.4.1` and `setuptools<81` are pinned for the same reasons as locally.)


In [ ]:
!pip -q install torch-geometric 'PyTDC==0.4.1' 'setuptools<81' rdkit scipy scikit-learn pyyaml
print('deps installed')


## 4. Download + preprocess DAVIS
Binarizes at Kd <= 30 nM and builds the stratified S1 split (same as locally).


In [ ]:
!python scripts/download_data.py --dataset davis
!python scripts/preprocess.py --dataset davis


## 5. Run the fusion ablation on GPU
Full DAVIS, all four fusions, seeds 42/43/44. On a T4 this is the main compute; expect roughly tens of minutes. To do a fast dry run first, add `--epochs 2 --max-batches 30`.


In [ ]:
!python scripts/run_ablation.py --dataset davis --split S1 --device cuda --seeds 42 43 44


## 6. Show + download results


In [ ]:
import json
res = json.load(open('experiments/results/ablation_davis_S1.json'))
print(json.dumps(res['summary'], indent=2))
print(json.dumps(res['significance_gca_vs_baselines'], indent=2))


In [ ]:
from google.colab import files
files.download('experiments/results/ablation_davis_S1.json')


## What the result means
- **AUPR** is the honest metric here (DAVIS is ~4% positive).
- RQ1 is supported if **gca** beats concat / flat_gate / ungated_xattn on AUPR, and the paired test `dAUPR` is positive with a small p-value.
- If GCA does *not* beat the simpler fusions, that is a real (publishable) finding too - see docs/objective1_analysis.md Risk R1.
